Calabines, Ayden Jarrick J.

# Activity 5.2 Topic Modeling

#### Objective(s):

This activity aims to introduce how to use LDA for topic modeling

#### Intended Learning Outcomes (ILOs):
* Demonstrate how to preprocess words in the dataset. 
* Demonstrate how to create and build LDA model using specified number of topics

#### Resources:
* Jupyter Notebook
* fetch20 dataset

#### Procedures
Load the necessary libraries and datasets

Remove the headers, footers, and quotes from each member of the set

In [1]:
import warnings
warnings.filterwarnings("ignore",category=DeprecationWarning)
import numpy as np
import nltk
import os
from sklearn import datasets

categories = ['alt.atheism', 'comp.graphics', 'rec.sport.baseball']
ng_train = datasets.fetch_20newsgroups(subset='train', 
                                       categories=categories, 
                                       remove=('headers', 
                                               'footers', 'quotes'))

In [2]:
print(ng_train.data[2])
print("++\n", ng_train.data[1504])
print("++\n", ng_train.data[1000])



	Sorry, I was, but I somehow have misplaced my diskette from the last 
couple of months or so. However, thanks to the efforts of Bobby, it is being 
replenished rather quickly!  

	Here is a recent favorite:

	--


       "Satan and the Angels do not have freewill.  
        They do what god tells them to do. "

        S.N. Mozumder (snm6394@ultb.isc.rit.edu) 


--


       "Satan and the Angels do not have freewill.  
        They do what god tells them to do. "
++
 

Why not use the PD C library for reading/writing TIFF files? It took me a
good 20 minutes to start using them in your own app.

Martin

--
---------------------------------------------------------------------------
++
 
Indeed, if the color teal on a team's uniforms is any indication of the
future, the Marlins are in dire trouble! Refer to the San Jose Sharks for
proof... But I have hope for the Marlins. I was a sometime member of the
Rene Lachemann fan club at the Oakland Coliseum, and have a deep respect
for the guy

* Pre-process all words in your document, including removing stop words.
* Remove words that show up in more than 60% of the documents/
* Vectorize your documents using NGrams

In [3]:
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer(ngram_range=(1, 2),  
                                   stop_words='english', 
                                   token_pattern="\\b[a-z][a-z]+\\b",
                                   lowercase=True,
                                   max_df = 0.6)

X = count_vectorizer.fit_transform(ng_train.data)

* Create an LDA model with 3 topics. You can do this with GenSim or SkLearn.
* Print out the topics and the 20 words most associated with that topic.
* Try using more or less topics, is there a sweet spot that allows us to separate out the three input classes?
* Find a document that is clearly about baseball, does the model choose it as dominantly the topic?
* Use pyLDAvis (pip install pyldavis) to create an interactive visualization of the topics

In [4]:
from sklearn.decomposition import LatentDirichletAllocation
n_topics = 4
n_iter = 10
lda = LatentDirichletAllocation(n_components=n_topics,
                                max_iter=n_iter,
                                random_state=42,
                               learning_method='online')
data = lda.fit_transform(X)
data[0]

array([0.00246896, 0.00251041, 0.99253159, 0.00248904])

In [5]:
print(ng_train.data[0]) # 99% composed of topic 3!




I happen to be a big fan of Jayson Stark.  He is a baseball writer for the 
Philadelphia Inquirer.  Every tuesday he writes a "Week in Review" column.  
He writes about unusual situations that occured during the week.  Unusual
stats.  He has a section called "Kinerisms of the Week" which are stupid
lines by Mets brodcaster Ralph Kiner.  Every year he has the LGTGAH contest.
That stands for "Last guy to get a hit."  He also writes for Baseball 
America.  That column is sort of a highlights of "Week in Review."  If you 
can, check his column out sometime.  He might make you laugh.

Rob Koffler



In [6]:
def display_topics(model, feature_names, no_top_words):
    for ix, topic in enumerate(model.components_):
        print("Topic ", ix)
        print(" ".join([feature_names[i]
                        for i in topic.argsort()[:-no_top_words - 1:-1]]))
        
display_topics(lda, count_vectorizer.get_feature_names_out(), 20)

Topic  0
jesus matthew said people den col prophecy int away war men messiah den den radius prophet row isaiah psalm row col sea
Topic  1
don god people does just think know like jpeg atheism say image time good believe way use atheists file religion
Topic  2
year game good team think don just games like players better runs hit won league time baseball season win pitching
Topic  3
graphics image edu data mail software ftp pub available send images package computer information use files thanks program processing code


* Open a new dataset from ap.txt
* Split this raw file into a set of documents. There is a clear marker between each article.
* Clean the text data and prepare for modeling (note that each document has some <XYZ> tags as well as extra spaces)

In [7]:
from pathlib import Path
import re

workspace_dir = Path.cwd()
candidate_paths = [
    workspace_dir / "ap.txt",
    workspace_dir / "ap_split.txt",
    workspace_dir / "dap_split.txt",
]

data_path = next((path for path in candidate_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("AP dataset file not found. Add ap.txt, ap_split.txt, or dap_split.txt to the notebook folder.")

with data_path.open("r", encoding="utf-8") as f:
    raw_text = f.read()

if data_path.name.lower() == "ap.txt":
    docs = [match.strip() for match in re.findall(r"<TEXT>\s*(.*?)\s*</TEXT>", raw_text, flags=re.DOTALL) if match.strip()]
else:
    docs = [doc.strip() for doc in raw_text.split("---") if doc.strip()]

print(f"Loaded {len(docs)} documents from {data_path.name}")
docs[1] if len(docs) > 1 else docs[0]

Loaded 2246 documents from ap.txt


"The Bechtel Group Inc. offered in 1985 to sell oil to Israel at a discount of at least $650 million for 10 years if it promised not to bomb a proposed Iraqi pipeline, a Foreign Ministry official said Wednesday. But then-Prime Minister Shimon Peres said the offer from Bruce Rappaport, a partner in the San Francisco-based construction and engineering company, was ``unimportant,'' the senior official told The Associated Press. Peres, now foreign minister, never discussed the offer with other government ministers, said the official, who spoke on condition of anonymity. The comments marked the first time Israel has acknowledged any offer was made for assurances not to bomb the planned $1 billion pipeline, which was to have run near Israel's border with Jordan. The pipeline was never built. In San Francisco, Tom Flynn, vice president for public relations for the Bechtel Group, said the company did not make any offer to Peres but that Rappaport, a Swiss financier, made it without Bechtel's k

In [8]:
import re

match = re.compile("<[^>]*>").search
for i, doc in enumerate(docs):
    final = []
    temp = doc.split('\n')
    for line in temp:
        if not match(line):
            final.append(line)
    docs[i] = ' '.join(final).strip().lower().replace("`", "").replace("'", "")

docs[1] if len(docs) > 1 else docs[0]

'the bechtel group inc. offered in 1985 to sell oil to israel at a discount of at least $650 million for 10 years if it promised not to bomb a proposed iraqi pipeline, a foreign ministry official said wednesday. but then-prime minister shimon peres said the offer from bruce rappaport, a partner in the san francisco-based construction and engineering company, was unimportant, the senior official told the associated press. peres, now foreign minister, never discussed the offer with other government ministers, said the official, who spoke on condition of anonymity. the comments marked the first time israel has acknowledged any offer was made for assurances not to bomb the planned $1 billion pipeline, which was to have run near israels border with jordan. the pipeline was never built. in san francisco, tom flynn, vice president for public relations for the bechtel group, said the company did not make any offer to peres but that rappaport, a swiss financier, made it without bechtels knowled

In [9]:
print(len(docs))

2246


Do LDA modeling to find topics in this chain of articles. Try many different numbers of topics and processing techniques. 

In [10]:
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer(ngram_range=(1, 3),
                                   stop_words='english',
                                   token_pattern="\\b[a-z][a-z]+\\b",
                                   lowercase=True,
                                   max_df=0.6)

X = count_vectorizer.fit_transform(docs)

In [11]:
from sklearn.decomposition import LatentDirichletAllocation

n_topics = 25
n_iter = 10
lda = LatentDirichletAllocation(n_components=n_topics,
                                max_iter=n_iter,
                                random_state=42,
                                learning_method='online')
data = lda.fit_transform(X)

In [12]:
def display_topics(model, feature_names, no_top_words, topic_names=None):
    for ix, topic in enumerate(model.components_):
        if not topic_names or not topic_names[ix]:
            print("\nTopic ", ix)
        else:
            print("\nTopic: '",topic_names[ix],"'")
        print(", ".join([feature_names[i]
                        for i in topic.argsort()[:-no_top_words - 1:-1]]))

display_topics(lda, count_vectorizer.get_feature_names_out(), 20)  # We have to look at the topics before hand and then add the labels afterwards)


Topic  0
bus, menem, greyhound, shuster, vietnam, driver, offenders, zaccaro, accident rate, wallenda, pekin, buses, gobie, drivers, joey, prom, algotson, probationers, kindergarten, broyles

Topic  1
sinhalese, fujimori, roberts, vargas, ferret, sri, gas, vargas llosa, llosa, oil, lima, shining, tamils, shining path, laurentiis, drilling, sri lanka, lanka, sinhalese extremists, tamil rebels

Topic  2
korean, north, korea, roh, kim, north korea, shuttle, north korean, inspection, nasa, united way, conable, mission, skunk, steinbergs, arco, redesigned, unification, seoul, challenger

Topic  3
fair, inches, rain, northern, snow, texas, temperatures, thunderstorms, ohio, degrees, showers, mississippi, southern, cloudy, valley, wiese, high, parts, plains, southwest

Topic  4
nbc, cbs, network, abc, maxwell, macmillan, rating, ratings, million, television, season, games, olympics, sports, tv, coverage, week, shows, movie, cable

Topic  5
soviet, party, government, gorbachev, communist, uni

In [13]:
tn = ["Political Media",None,"Financials",None,"Nordstrom Scandal","Oil","Hurricanes","North Korea","NASA","US Politics","TV Networks","Forest Fires",
      None,"Agriculture/Drought","Middle East","US Political Campaigns","Pollution","Carribean","Health/Medical","Theatre/Arts","Global Warming",
      "Advertisements","Southern US Weather","South America",None]
display_topics(lda, count_vectorizer.get_feature_names_out(), 20, topic_names=tn)


Topic: ' Political Media '
bus, menem, greyhound, shuster, vietnam, driver, offenders, zaccaro, accident rate, wallenda, pekin, buses, gobie, drivers, joey, prom, algotson, probationers, kindergarten, broyles

Topic  1
sinhalese, fujimori, roberts, vargas, ferret, sri, gas, vargas llosa, llosa, oil, lima, shining, tamils, shining path, laurentiis, drilling, sri lanka, lanka, sinhalese extremists, tamil rebels

Topic: ' Financials '
korean, north, korea, roh, kim, north korea, shuttle, north korean, inspection, nasa, united way, conable, mission, skunk, steinbergs, arco, redesigned, unification, seoul, challenger

Topic  3
fair, inches, rain, northern, snow, texas, temperatures, thunderstorms, ohio, degrees, showers, mississippi, southern, cloudy, valley, wiese, high, parts, plains, southwest

Topic: ' Nordstrom Scandal '
nbc, cbs, network, abc, maxwell, macmillan, rating, ratings, million, television, season, games, olympics, sports, tv, coverage, week, shows, movie, cable

Topic: ' O


The topic modeling results show that LDA was able to separate the 20 Newsgroups dataset into meaningful themes based on the most frequent words in each topic. One topic is strongly associated with religion and atheism, another is centered on baseball terms, and another is clearly related to computer graphics and software. This shows that preprocessing, vectorization, and LDA were effective in identifying the dominant subject of each group of documents.ry Activity

## Suplementary Activity

* Use your own dataset
* Perform preprocessing of words in the dataset
* Create LDA model using a specified number of topics

In [1]:
import kagglehub
from pathlib import Path

# Download the movie review dataset from Kaggle.
path = Path(kagglehub.dataset_download("nltkdata/movie-review"))

print("Path to dataset files:", path)

c:\Users\Mong\Desktop\Eden\MASK_RCNN\SolDef_AI\.venv\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


c:\Users\Mong\Desktop\Eden\MASK_RCNN\SolDef_AI\.venv\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\Mong\.cache\kagglehub\datasets\nltkdata\movie-review\versions\3


In [2]:
# Collect all review text files from the downloaded dataset.
all_txt_files = sorted(path.rglob("*.txt"))

print("Total text files:", len(all_txt_files))
print("Sample files:")
for sample_path in all_txt_files[:5]:
    print(sample_path)

Total text files: 2000
Sample files:
C:\Users\Mong\.cache\kagglehub\datasets\nltkdata\movie-review\versions\3\movie_reviews\movie_reviews\neg\cv000_29416.txt
C:\Users\Mong\.cache\kagglehub\datasets\nltkdata\movie-review\versions\3\movie_reviews\movie_reviews\neg\cv001_19502.txt
C:\Users\Mong\.cache\kagglehub\datasets\nltkdata\movie-review\versions\3\movie_reviews\movie_reviews\neg\cv002_17424.txt
C:\Users\Mong\.cache\kagglehub\datasets\nltkdata\movie-review\versions\3\movie_reviews\movie_reviews\neg\cv003_12683.txt
C:\Users\Mong\.cache\kagglehub\datasets\nltkdata\movie-review\versions\3\movie_reviews\movie_reviews\neg\cv004_12641.txt


In [3]:
# Load the review text and keep the folder name as a simple label.
docs = []
labels = []

for file_path in all_txt_files:
    text = file_path.read_text(encoding="utf-8", errors="ignore").strip()
    if text:
        docs.append(text)
        labels.append(file_path.parent.name)

print("Loaded documents:", len(docs))
print("Labels found:", sorted(set(labels)))
print(docs[0][:1000])

Loaded documents: 2000
Labels found: ['neg', 'pos']
plot : two teen couples go to a church party , drink and then drive . 
they get into an accident . 
one of the guys dies , but his girlfriend continues to see him in her life , and has nightmares . 
what's the deal ? 
watch the movie and " sorta " find out . . . 
critique : a mind-fuck movie for the teen generation that touches on a very cool idea , but presents it in a very bad package . 
which is what makes this review an even harder one to write , since i generally applaud films which attempt to break the mold , mess with your head and such ( lost highway & memento ) , but there are good and bad ways of making all types of films , and these folks just didn't snag this one correctly . 
they seem to have taken this pretty neat concept , but executed it terribly . 
so what are the problems with the movie ? 
well , its main problem is that it's simply too jumbled . 
it starts off " normal " but then downshifts into this " fantasy " wor

In [4]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Normalize text and remove punctuation and common stop words.
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = [word for word in text.split() if len(word) > 2 and word not in ENGLISH_STOP_WORDS]
    return " ".join(tokens)


clean_docs = [preprocess_text(doc) for doc in docs]
print(clean_docs[0][:1000])

plot teen couples church party drink drive accident guys dies girlfriend continues life nightmares deal watch movie sorta critique mind fuck movie teen generation touches cool idea presents bad package makes review harder write generally applaud films attempt break mold mess head lost highway memento good bad ways making types films folks just didn snag correctly taken pretty neat concept executed terribly problems movie main problem simply jumbled starts normal downshifts fantasy world audience member idea going dreams characters coming dead look like dead strange apparitions disappearances looooot chase scenes tons weird things happen simply explained personally don mind trying unravel film does clue kind fed film biggest problem obviously got big secret hide want hide completely final minutes make things entertaining thrilling engaging meantime really sad arrow dig flicks like actually figured half way point strangeness did start make little bit sense didn make film entertaining gue

In [5]:
from sklearn.feature_extraction.text import CountVectorizer

# Convert the cleaned reviews into a document-term matrix.
movie_vectorizer = CountVectorizer(ngram_range=(1, 2),
                                   stop_words='english',
                                   token_pattern=r"\b[a-z][a-z]+\b",
                                   lowercase=True,
                                   max_df=0.6,
                                   min_df=5)

movie_X = movie_vectorizer.fit_transform(clean_docs)
print(movie_X.shape)

(2000, 19752)


In [6]:
from sklearn.decomposition import LatentDirichletAllocation

# Train an LDA model with a chosen number of topics.
movie_n_topics = 6
movie_lda = LatentDirichletAllocation(n_components=movie_n_topics,
                                      max_iter=10,
                                      random_state=42,
                                      learning_method='online')
movie_topic_matrix = movie_lda.fit_transform(movie_X)
movie_topic_matrix[0]

array([5.20296564e-04, 4.61036199e-02, 5.18653018e-04, 5.14492268e-04,
       5.15340426e-04, 9.51827598e-01])

In [20]:
# Print the most important words associated with each topic.
def display_movie_topics(model, feature_names, no_top_words):
    for topic_index, topic in enumerate(model.components_):
        print(f"Topic {topic_index}")
        print(", ".join(feature_names[i] for i in topic.argsort()[:-no_top_words - 1:-1]))
        print()


movie_feature_names = movie_vectorizer.get_feature_names_out()
display_movie_topics(movie_lda, movie_feature_names, 15)

Topic 0
story, good, action, effects, world, films, new, does, alien, scenes, character, special, make, people, life

Topic 1
scream, horror, smith, man, good, kevin, joe, director, tarantino, know, killer, carter, tarzan, movies, little

Topic 2
disney, shrek, carpenter, bacon, wild things, joe, vampires, wild, mermaid, finn, sleepy, hollow, princess, little mermaid, stuart

Topic 3
truman, carrey, jim carrey, marie, carry, jim, gattaca, paulie, irene, ned, kenneth, ace, niccol, sid, burbank

Topic 4
wars, star wars, star, lucas, phantom, menace, phantom menace, jedi, jar, effects, special, spawn, wan, trilogy, obi wan

Topic 5
good, character, story, characters, way, life, make, really, little, does, scene, people, plot, love, films



In [7]:
import pandas as pd

# Summarize the dominant topic assigned to each review.
movie_results = pd.DataFrame({
    "label": labels,
    "dominant_topic": movie_topic_matrix.argmax(axis=1),
    "topic_score": movie_topic_matrix.max(axis=1),
    "preview": [doc[:200] for doc in docs]
})

movie_results.head(10)

,label,dominant_topic,topic_score,preview
0,neg,5,0.951828,"plot : two teen couples go to a church party ,..."
1,neg,0,0.622033,the happy bastard's quick movie review \ndamn ...
2,neg,5,0.953466,it is movies like these that make a jaded movi...
3,neg,0,0.742882,""" quest for camelot "" is warner bros . ' first..."
4,neg,1,0.643794,synopsis : a mentally unstable man undergoing ...
5,neg,0,0.636641,capsule : in 2176 on the planet mars police ta...
6,neg,1,0.658494,"so ask yourself what "" 8mm "" ( "" eight millime..."
7,neg,5,0.828621,that's exactly how long the movie felt to me ....
8,neg,5,0.869670,call it a road trip for the walking wounded . ...
9,neg,5,0.813820,plot : a young french boy sees his parents kil...


In [22]:
label_topic_strength = pd.DataFrame(movie_topic_matrix, columns=[f"topic_{i}" for i in range(movie_n_topics)])
label_topic_strength["label"] = labels

label_topic_strength.groupby("label").mean().round(3)

,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5
label,,,,,,
neg,0.228,0.090,0.004,0.005,0.003,0.669
pos,0.255,0.055,0.006,0.005,0.008,0.670


In [23]:
for topic_id in range(movie_n_topics):
    print(f"Representative reviews for Topic {topic_id}")
    top_docs = movie_results[movie_results["dominant_topic"] == topic_id].nlargest(3, "topic_score")
    for _, row in top_docs.iterrows():
        print(f"Label: {row['label']} | Score: {row['topic_score']:.3f}")
        print(row["preview"])
        print()
    print("-" * 80)

Representative reviews for Topic 0
Label: neg | Score: 0.999
i think that saying that the x-files is one of this summer's most anticipated films is safe . 
for five years , " the x-files " television show has developed a dedicated fan culture , whose rabid devo

Label: pos | Score: 0.999
with the success of the surprise hit alien , directed by ridley scott , a sequel was inevitable . 
in fact , after watching the first film , a sequel was wanted , particularly by this reviewer . 
hand

Label: neg | Score: 0.999
over 40 years ago , a japanese production company called toho introduced the land of the rising sun to gojira , a reptilian creature of immense proportions created by mankind's nuclear testing . 
part

--------------------------------------------------------------------------------
Representative reviews for Topic 1
Label: neg | Score: 0.998
capsule : gal is a 50s-ish london cockney gangster who has retired to spain . 
his old associates want him for one last job and send the vi

In [8]:
topic_by_label = pd.crosstab(
    movie_results["label"],
    movie_results["dominant_topic"],
    normalize="index"
).round(3)

topic_by_label

dominant_topic,0,1,3,4,5
label,,,,,
neg,0.193,0.060,0.001,0.000,0.746
pos,0.222,0.018,0.000,0.008,0.752


#### Conclusion

The movie review dataset was preprocessed and analyzed using LDA topic modeling. The model identified meaningful themes in the reviews such as plot, acting, and overall movie quality, based on the most common words within each topic. The results show that topic modeling can reveal hidden patterns in text, although the discovered topics describe general review themes more than a strict separation between positive and negative sentiment.